<a href="https://colab.research.google.com/github/SantiAlfonso/Modelo_con_Scikit-learn/blob/main/Modelo_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Objetivos

*   Construir un modelo de regresión lineal que permita estimar el área que podría verse afectada por un incendio forestal.

*   Determinar cuáles son las variables que más inciden en la magnitud de un incendio.






# Instalaer librerías externas

In [86]:
!pip install -U scikit-learn
!pip install statsmodels

# Importar librerías

In [87]:
# Librería para comandos del sistema
import os
# librería para manejo de datos
import pandas as pd
import numpy as np
# Librería para aprendizaje automatico
# Para realizar la separación del conjunto de aprendizaje en entrenamiento y test.
from sklearn.model_selection import train_test_split
# Para conbstruir un modelo con el algoritmo de regresión lineal
from sklearn.linear_model import LinearRegression
# Para determinar el rendimiento del modelo con las métricas MSE, MAE y R2
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# Para sacar un reporte estadístico que podemos uysar para determinar la importancia de las variables explicativas
import statsmodels.api as sm

# Cargar datos

In [88]:
data = pd.read_excel('incendio_forestal.xlsx')

observamos la cantidad de filas y columnas que posee el dataframe

In [89]:
data.describe()

,area_quemada,Prec_pre_30,Prec_pre_15,Prec_pre_7,Prec_cont
count,37759.000000,37759.000000,37759.000000,37759.000000,37759.000000
mean,2347.020001,35.931177,16.127596,6.764959,21.260950
std,15513.934516,130.257343,65.928608,36.746405,69.377046
min,0.510000,0.000000,0.000000,0.000000,0.000000
25%,1.200000,0.000000,0.000000,0.000000,0.000000
50%,4.000000,0.600000,0.000000,0.000000,0.000000
75%,20.000000,37.900000,12.300000,1.500000,3.300000
max,538049.000000,13560.800000,2527.000000,1638.000000,2126.000000


In [90]:
data.shape

(37759, 8)

In [91]:
data.head()

,area_quemada,clase_incendio,mes_incendio,vegetacion,Prec_pre_30,Prec_pre_15,Prec_pre_7,Prec_cont
0,3.0,B,Diciembre,Desierto polar_roca,59.8,8.4,0.0,86.8
1,60.0,C,Febrero,Bosque tropical perennifolio secundario,168.8,42.2,18.1,124.5
2,1.0,B,Junio,Bosque tropical perennifolio,10.4,7.2,0.0,0.0
3,5.2,B,Enero,Matorral abierto,26.0,0.0,0.0,0.0
4,1.0,B,Noviembre,Matorral abierto,28.4,27.5,1.2,55.4


# Preparamos los datos

Antes de realizar cualquier paso es importante asegurar que estos datos no contienen errores, como datos faltantes, duplicados.

In [92]:
!pip install -U ydata-profiling


In [93]:
import pandas_profiling
pandas_profiling.ProfileReport(data)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 8/8 [00:00<00:00, 23.69it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

*   En el análisis observamos que tiene 1929 filas duplicadas.
*   Las variables de precipitación tienen una alta correlación entre ellas, lo que puede generar problemas de multicolinealidad.
* También observamos un alto numero de ceros en las variables de precipitaciones




In [94]:
data.shape

(37759, 8)

In [95]:
# Creamos otro dataframe para no trabajar sobre el original
data_t = data.copy()
# 1. Quitamos los duplicados
data_t = data_t.drop_duplicates()
# 2. Nos quedamos solo con la lluvia menos problemática y transformar su sesgo
data_t['Prec_pre_30_log'] = np.log1p(data_t['Prec_pre_30'])
# 3. Eliminar las variables colineales y con exceso de ceros
data_t = data_t.drop(columns=['Prec_pre_15', 'Prec_pre_7', 'Prec_cont', 'Prec_pre_30'])

Tenemos que tener en cuenta que los requerimientos de entrada de los algoritmos de aprendizaje. Para el algoritmo de regresión lineal todas las variables deben ser numéricas, pero en nuestro set de datos poseemos variables categoricas , por lo que se debe realizar una transformación de estas a un formato numerico que se conoce como variables dummies.

In [96]:
# Se muestran las categorías de la variable "clase_incendio" con sus frecuencias
pd.value_counts(data['clase_incendio'])

/tmp/ipykernel_634/3965746002.py:2: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(data['clase_incendio'])


,count
clase_incendio,
B,24554
C,7363
G,2997
F,1426
D,954
E,465


In [97]:
# Se muestran las categorías de la variable "mes_incendio" con sus frecuencias
pd.value_counts(data['mes_incendio'])

/tmp/ipykernel_634/3711845707.py:2: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(data['mes_incendio'])


,count
mes_incendio,
Marzo,5241
Abril,5003
Julio,4253
Agosto,4087
Junio,3485
Febrero,3130
Mayo,2943
Septiembre,2359
Enero,2156


In [98]:
# Se muestran las categorías de la variable "vegetacion" con sus frecuencias
pd.value_counts(data['vegetacion'])

/tmp/ipykernel_634/1451999554.py:2: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(data['vegetacion'])


,count
vegetacion,
Matorral abierto,11949
Bosque tropical perennifolio secundario,8087
Desierto polar_roca,7510
Bosque tropical perennifolio,6531
Pradera_estepa,2618
Bosque templado,619
Desierto,445


In [99]:
# Se realiza la transformación de estas variables a dummies.
# Al agregar dtype=int, las columnas nacerán como 1 y 0 numéricos
data_t = pd.get_dummies(data_t, columns=['clase_incendio', 'mes_incendio', 'vegetacion'], drop_first=True, dtype=int)

In [100]:
# Cantidad de datos y número de variables después de esta transformación.
data_t.shape

(26394, 24)

In [101]:
data_t.head()

,area_quemada,Prec_pre_30_log,clase_incendio_C,clase_incendio_D,clase_incendio_E,clase_incendio_F,clase_incendio_G,mes_incendio_Agosto,mes_incendio_Diciembre,mes_incendio_Enero,...,mes_incendio_Mayo,mes_incendio_Noviembre,mes_incendio_Octubre,mes_incendio_Septiembre,vegetacion_Bosque tropical perennifolio,vegetacion_Bosque tropical perennifolio secundario,vegetacion_Desierto,vegetacion_Desierto polar_roca,vegetacion_Matorral abierto,vegetacion_Pradera_estepa
0,3.0,4.107590,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
1,60.0,5.134621,1,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,1.0,2.433613,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,5.2,3.295837,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
4,1.0,3.380995,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0


# Construcción del modelo

Los algoritmos supervisados immplementados con la librería scikit-learn requieren que las variables de entrada estén separadas de la variable objetivo. En el casoa de este modelo, la variable objetivo es el "area_quemada".

In [102]:
Y = data_t['area_quemada']
X = data_t.drop(['area_quemada'],axis=1)

In [103]:
X.head()

,Prec_pre_30_log,clase_incendio_C,clase_incendio_D,clase_incendio_E,clase_incendio_F,clase_incendio_G,mes_incendio_Agosto,mes_incendio_Diciembre,mes_incendio_Enero,mes_incendio_Febrero,...,mes_incendio_Mayo,mes_incendio_Noviembre,mes_incendio_Octubre,mes_incendio_Septiembre,vegetacion_Bosque tropical perennifolio,vegetacion_Bosque tropical perennifolio secundario,vegetacion_Desierto,vegetacion_Desierto polar_roca,vegetacion_Matorral abierto,vegetacion_Pradera_estepa
0,4.107590,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0
1,5.134621,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,1,0,0,0,0
2,2.433613,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,3.295837,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
4,3.380995,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0


In [104]:
Y.head()

,area_quemada
0,3.0
1,60.0
2,1.0
3,5.2
4,1.0


A continuación debemos hacer la separación de nuestros datos en un conjunto para entrenamiento y otro para el test.

El conjunto de entrenamiento se utiliza para ajustar (entrenar) un modelo. Luego, se utiliza el conjunto test para hacer predicciones, las cuales se comparan con los valores esperados para determinar su rendimiento utilizando la métrica seleccionada.

**Train data:** se utiliza para entrenar el modelo con el algoritmo de aprendizaje.

**Test data:** se utiliza para evaluar el ajuste del modelo.

In [105]:
# Se realiza la división entrenamiento - test. Se deja 20% de los datos para el test.
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2,random_state=0)

Antes de construir el modelo debemos crear un objeto de la clase LinearRegression.

In [106]:
# Primero se crea el objeto para construir el modelo
modelo_regresion = LinearRegression()
# Podemos verificar que lo hemos construido.
modelo_regresion

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


A continuación, procedemos a entrenar el modelo utilizando el conjunto de entrenamiento.

In [107]:
# Ajustar el modelo con los datos de entrenamiento
modelo_regresion.fit(X_train,Y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](23,)","[ -106.38, -99.33, -211.3 ,...,-1173.93, -993.8 , -913.64]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](23,)","['Prec_pre_30_log','clase_incendio_C','clase_incendio_D',..., 'vegetacion_Desierto polar_roca','vegetacion_Matorral abierto', 'vegetacion_Pradera_estepa']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,997.5
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,23
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,23


# Evaluamos el modelo

Las metricas de evaluación nos van a permitir determinar qué tan bien se desempeña un modelo; es decir, cómo se ajusta a los datos.

Mean-Squared-Error(MSE). Error medio cuadrático

Mean-Absolute-Error(MAE). Error absoluto medio

R² or Coeficiente de determinación.

In [108]:
y_pred = modelo_regresion.predict(X_train)
# Se obtienen las metricas a partir de la predicción y la base de evaluación (valores reales).
print("-- METRICAS DE ENTRENAMIENTO --")
print("RMSE: %.2f" % np.sqrt(mean_squared_error(Y_train,y_pred)))
print("MAE: %.2f" % mean_absolute_error(Y_train, y_pred))
print('R²: %.2f' % r2_score(Y_train, y_pred))

-- METRICAS DE ENTRENAMIENTO --
RMSE: 15312.18
MAE: 3165.68
R²: 0.23


In [109]:
y_pred = modelo_regresion.predict(X_test)
# Se obtienen las metricas a partir de la predicción y la base de evaluación (valores reales).
print("-- METRICAS DE ENTRENAMIENTO --")
print("RMSE: %.2f" % np.sqrt(mean_squared_error(Y_test,y_pred)))
print("MAE: %.2f" % mean_absolute_error(Y_test, y_pred))
print('R²: %.2f' % r2_score(Y_test, y_pred))

-- METRICAS DE ENTRENAMIENTO --
RMSE: 17240.04
MAE: 3655.67
R²: 0.23


In [110]:
# Ajustar el modelo con los datos de entrenamiento
modelo_regresion.fit(X,Y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](23,)","[-110.14,-102.18,-221.33,...,-644. ,-419.72,-588.56]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](23,)","['Prec_pre_30_log','clase_incendio_C','clase_incendio_D',..., 'vegetacion_Desierto polar_roca','vegetacion_Matorral abierto', 'vegetacion_Pradera_estepa']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,672.6
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,23
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,23


In [111]:
# Podemos visualizar los parámetros del modelos (coeficientes de regresión)
modelo_regresion.coef_

array([ -110.14370684,  -102.17669384,  -221.32682054,   187.90782842,
        2986.88449665, 26978.81629396,  -576.51171102,  -206.37409891,
        -278.10877683,  -136.77505163,    85.67805235,  1982.2640867 ,
        -199.38367955,   382.06441132,  -483.23609963,  -242.65504728,
        -717.61886305,  1046.75754591,  -452.68940977, -2424.83149632,
        -643.99898905,  -419.72344444,  -588.56143016])

In [112]:
# Ajustar el modelo para ver el reporte
model = sm.OLS(Y, X).fit() ## sm.OLS(output, input)
# Mostrar las estadísticas del modelo
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:           area_quemada   R-squared (uncentered):                   0.254
Model:                            OLS   Adj. R-squared (uncentered):              0.253
Method:                 Least Squares   F-statistic:                              390.6
Date:                Mon, 15 Jun 2026   Prob (F-statistic):                        0.00
Time:                        17:22:17   Log-Likelihood:                     -2.9248e+05
No. Observations:               26394   AIC:                                  5.850e+05
Df Residuals:                   26371   BIC:                                  5.852e+05
Df Model:                          23                                                  
Covariance Type:            nonrobust                                                  
======================================================================================================================
                                                         coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------
Prec_pre_30_log                                      -97.1793     52.422     -1.854      0.064    -199.930       5.572
clase_incendio_C                                     -80.8885    249.597     -0.324      0.746    -570.112     408.334
clase_incendio_D                                    -191.1767    552.971     -0.346      0.730   -1275.031     892.677
clase_incendio_E                                     224.0560    764.714      0.293      0.770   -1274.824    1722.936
clase_incendio_F                                    3027.2511    464.845      6.512      0.000    2116.129    3938.373
clase_incendio_G                                    2.702e+04    356.481     75.784      0.000    2.63e+04    2.77e+04
mes_incendio_Agosto                                 -481.5533    387.915     -1.241      0.214   -1241.887     278.780
mes_incendio_Diciembre                              -130.0019    610.830     -0.213      0.831   -1327.261    1067.258
mes_incendio_Enero                                  -195.8350    523.149     -0.374      0.708   -1221.236     829.566
mes_incendio_Febrero                                 -64.8314    440.199     -0.147      0.883    -927.645     797.982
mes_incendio_Julio                                   177.4189    386.576      0.459      0.646    -580.290     935.128
mes_incendio_Junio                                  2066.3606    416.708      4.959      0.000    1249.590    2883.132
mes_incendio_Marzo                                  -124.9079    372.990     -0.335      0.738    -855.988     606.172
mes_incendio_Mayo                                    471.5802    427.982      1.102      0.271    -367.288    1310.449
mes_incendio_Noviembre                              -402.1168    503.497     -0.799      0.425   -1388.997     584.764
mes_incendio_Octubre                                -158.6970    478.283     -0.332      0.740   -1096.157     778.763
mes_incendio_Septiembre                             -632.3380    456.264     -1.386      0.166   -1526.641     261.965
vegetacion_Bosque tropical perennifolio             1598.8734    380.689      4.200      0.000     852.703    2345.044
vegetacion_Bosque tropical perennifolio secundario   103.3002    365.700      0.282      0.778    -613.492     820.092
vegetacion_Desierto                                -1861.7905    858.208     -2.169      0.030   -3543.924    -179.657
vegetacion_Desierto polar_roca                       -87.0932    351.512     -0.248      0.804    -776.075     601.889
vegetacion_Matorral abierto                          134.8313    350.590      0.385      0.701    -552.344     822.007
vegetacio